In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             fbeta_score, roc_auc_score, confusion_matrix, precision_recall_curve)
df = pd.read_csv("train.csv")

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [46]:
print(df['Survived'].value_counts(normalize=True))

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


In [3]:
X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
y = df['Survived']

In [4]:
X['Age'] = X['Age'].fillna(X['Age'].median())
X['Fare'] = X['Fare'].fillna(X['Fare'].median())
X['Embarked'] = X['Embarked'].fillna(X['Embarked'].mode()[0])

/tmp/ipykernel_1465/2507831288.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Age'] = X['Age'].fillna(X['Age'].median())
/tmp/ipykernel_1465/2507831288.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Fare'] = X['Fare'].fillna(X['Fare'].median())
/tmp/ipykernel_1465/2507831288.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/p

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=2023, test_size=0.25, stratify=y)
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
805,3,male,31.0,0,0,7.7750,S
55,1,male,28.0,0,0,35.5000,S
325,1,female,36.0,0,0,135.6333,C
543,2,male,32.0,1,0,26.0000,S
556,1,female,48.0,1,0,39.6000,C


In [6]:
num = ['Age', 'Fare', 'SibSp', 'Parch']
column = ['Pclass', 'Sex', 'Embarked']
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num),
    ('column', OneHotEncoder(drop='first', sparse_output=False), column)
])

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Age', 'Fare', 'SibSp',
                                                   'Parch']),
                                                 ('column',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['Pclass', 'Sex',
                                                   'Embarked'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=2023))])

In [21]:
dummy_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent'))
])
dummy_pipe.fit(X_train, y_train)
y_pred_dummy = dummy_pipe.predict(X_test)

In [42]:
print("Константный бейзлайн:")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_dummy):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_dummy, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_dummy):.4f}")
print(f"F1-score  : {f1_score(y_test, y_pred_dummy):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_dummy))

Константный бейзлайн:
Accuracy  : 0.6143
Precision : 0.0000
Recall    : 0.0000
F1-score  : 0.0000
Confusion matrix:
 [[137   0]
 [ 86   0]]


In [41]:
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=2023))
])
pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Age', 'Fare', 'SibSp',
                                                   'Parch']),
                                                 ('column',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['Pclass', 'Sex',
                                                   'Embarked'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=2023))])

In [39]:
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print("Логистическая регрессия:")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1-score  : {f1_score(y_test, y_pred):.4f} (основная метрика)")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Логистическая регрессия:
Accuracy  : 0.7534
Precision : 0.6703
Recall    : 0.7093
F1-score  : 0.6893 (основная метрика)
ROC-AUC   : 0.8252
Confusion matrix:
 [[107  30]
 [ 25  61]]


In [40]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1 = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='f1')
print("Кросс-валидация (5 фолдов) – средний F1: {:.4f} (+/- {:.4f})".format(cv_f1.mean(), cv_f1.std()))

Кросс-валидация (5 фолдов) – средний F1: 0.7449 (+/- 0.0596)


Для оценки качества модели выбрана метрика F1-score. Она усредняет precision и recall, обеспечивает объективную оценку при дисбалансе классов - 38% (доля выживших (класс 1)) и 62% (доля погибших (класс 0)). F1-score учитывает баланс между ложноположительными и ложноотрицательными прогнозами.